In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [11]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
spark.version

'4.1.1'

In [5]:
df = spark.read.\
    parquet('yellow_tripdata_2025-11.parquet')\
    .repartition(4)

In [14]:
df.write.parquet('yellow_tripdata/2025/pq/')

In [2]:
!ls -lh yellow_tripdata/2025/pq

total 102M
-rw-r--r-- 1 codespace codespace   0 Mar 24 22:00 _SUCCESS
-rw-r--r-- 1 codespace codespace 26M Mar 24 22:00 part-00000-5dda0b96-10b1-46d5-91c5-78a90cbd529d-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar 24 22:00 part-00001-5dda0b96-10b1-46d5-91c5-78a90cbd529d-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar 24 22:00 part-00002-5dda0b96-10b1-46d5-91c5-78a90cbd529d-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar 24 22:00 part-00003-5dda0b96-10b1-46d5-91c5-78a90cbd529d-c000.snappy.parquet


In [6]:
df = spark.read.\
    parquet('yellow_tripdata_2025-11.parquet')

In [7]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [13]:
trips_15_nov = df.filter(F.to_date("tpep_pickup_datetime") == F.lit("2025-11-15"))
count_df = trips_15_nov.count()
print(count_df)

162604


In [ ]:
df.registerTempTable('trips_data')

In [32]:
df_result = spark.sql("""
SELECT 
    MAX(
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600
    ) AS longest_trip_hours
FROM trips_data
""")
df_result.show()

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



In [33]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-25 09:06:26--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.212, 18.239.238.152, 18.239.238.119, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.212|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-25 09:06:26 (1.52 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [36]:

df_zone_lookup = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

df_zone_lookup.createOrReplaceTempView('zone_lookup')

In [42]:
df_least_frequent = spark.sql("""
SELECT
    z.Zone,
    COUNT(*) AS pickup_count
FROM
    trips_data t
JOIN
    zone_lookup z ON t.PULocationID = z.LocationID
GROUP BY
    z.Zone
ORDER BY
    pickup_count ASC
LIMIT 5
""")

df_least_frequent.show()

+--------------------+------------+
|                Zone|pickup_count|
+--------------------+------------+
|       Arden Heights|           1|
|Eltingville/Annad...|           1|
|Governor's Island...|           1|
|       Port Richmond|           3|
|       Rikers Island|           4|
+--------------------+------------+

